# Notatnik 2 ? Regresja California Housing z Linear Regression i Ridge

Tym razem przewidujemy liczb?: median? warto?ci dom?w w danym obszarze Kalifornii. To problem regresji.

Wa?na r??nica:

```text
klasyfikacja ? przewidujemy kategori?
regresja     ? przewidujemy liczb?
```

## 1. Wczytanie danych

`fetch_california_housing()` mo?e przy pierwszym uruchomieniu pobra? dane z internetu i zapisa? je lokalnie.

In [ ]:
from sklearn.datasets import fetch_california_housing
import pandas as pd

housing = fetch_california_housing(as_frame=True)
df = housing.frame

print("California Housing za?adowany.")
print("Liczba wierszy:", len(df))
print("Liczba kolumn:", len(df.columns))

df.head()

## 2. Znaczenie kolumn

Ka?dy wiersz opisuje obszar, nie pojedynczy dom. `MedHouseVal` jest celem i oznacza warto?? w setkach tysi?cy dolar?w.

In [ ]:
print(df.describe().T[["mean", "std", "min", "max"]])

## 3. Dzielimy dane na cechy i target

In [ ]:
X = housing.data
y = housing.target

print("Cechy:", list(X.columns))
print("Target:", housing.target_names[0])
print("X shape:", X.shape)
print("y shape:", y.shape)

## 4. Podzia? train/test

W regresji nie u?yjemy prostego `stratify=y`, bo `y` jest liczb? ci?g??. Na tym poziomie wystarczy losowy, powtarzalny podzia?.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)

## 5. Funkcja do oceny regresji

Jedna metryka rzadko wystarcza. U?yjemy:

- MAE: przeci?tny b??d bez znaku,
- RMSE: mocniej karze du?e pomy?ki,
- R?: ile zmienno?ci danych model potrafi wyja?ni?.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


def evaluate_regression(name, model, X_train, y_train, X_test, y_test):
    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)

    result = {
        "model": name,
        "train_MAE": mean_absolute_error(y_train, train_pred),
        "test_MAE": mean_absolute_error(y_test, test_pred),
        "test_RMSE": mean_squared_error(y_test, test_pred) ** 0.5,
        "test_R2": r2_score(y_test, test_pred),
    }
    return result


def dollars(value):
    return f"{value * 100_000:,.0f} USD"

## 6. Baseline: ?rednia warto??

`DummyRegressor` przewiduje zawsze ?redni? z danych treningowych. Je?li model nie bije tego baseline'u, to znaczy, ?e jeszcze nic sensownego si? nie nauczy?.

In [ ]:
from sklearn.dummy import DummyRegressor

baseline = DummyRegressor(strategy="mean")
baseline.fit(X_train, y_train)

baseline_result = evaluate_regression(
    "DummyRegressor(mean)", baseline, X_train, y_train, X_test, y_test
)

baseline_result

## 7. Linear Regression

Regresja liniowa pr?buje opisa? wynik jako sum? wp?yw?w poszczeg?lnych cech. Jest szybka i interpretowalna, ale mo?e by? zbyt prosta dla nieliniowych danych.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

linear_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LinearRegression()),
])

linear_model.fit(X_train, y_train)
linear_result = evaluate_regression(
    "LinearRegression", linear_model, X_train, y_train, X_test, y_test
)

linear_result

## 8. Ridge Regression

Ridge to regresja liniowa z regularyzacj?. Regularyzacja ogranicza wielko?? wsp??czynnik?w, co cz?sto pomaga modelowi generalizowa?.

In [ ]:
from sklearn.linear_model import Ridge

ridge_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Ridge(alpha=1.0)),
])

ridge_model.fit(X_train, y_train)
ridge_result = evaluate_regression(
    "Ridge(alpha=1.0)", ridge_model, X_train, y_train, X_test, y_test
)

ridge_result

## 9. Por?wnanie modeli

Patrzymy nie tylko na to, kt?ry wynik jest najlepszy, ale te? czy r??nica jest du?a i czy model jest du?o lepszy od baseline'u.

In [ ]:
results = pd.DataFrame([baseline_result, linear_result, ridge_result])
results["test_MAE_USD"] = results["test_MAE"].map(dollars)
results["test_RMSE_USD"] = results["test_RMSE"].map(dollars)

results.sort_values("test_MAE")

## 10. Wsp??czynniki modelu Ridge

W modelach liniowych mo?emy zobaczy? wsp??czynniki. Po skalowaniu cech daj? one orientacyjny obraz, kt?re zmienne silniej wp?ywaj? na predykcj?.

In [ ]:
coefficients = pd.Series(
    ridge_model.named_steps["model"].coef_,
    index=X.columns,
).sort_values()

coefficients.plot(kind="barh", title="Wsp??czynniki Ridge po skalowaniu")
plt = __import__("matplotlib.pyplot").pyplot
plt.xlabel("coefficient")
plt.show()

coefficients.sort_values(key=abs, ascending=False)

## 11. Predykcja dla jednego obszaru

Por?wnajmy predykcj? z prawdziw? warto?ci?. To cz?sto lepiej dzia?a na wyobra?ni? ni? sama tabela metryk.

In [ ]:
example_id = 10
example = X_test.iloc[[example_id]]
true_value = y_test.iloc[example_id]
predicted_value = ridge_model.predict(example)[0]

print("Prawdziwa warto??:", dollars(true_value))
print("Predykcja Ridge:   ", dollars(predicted_value))
print("B??d:              ", dollars(abs(true_value - predicted_value)))

example

## Twoja kolej

1. Zmie? `example_id` na inn? warto??.
2. Por?wnaj b??d modelu dla r??nych obszar?w.
3. Sprawd?, czy s? przyk?ady, na kt?rych model myli si? bardzo mocno.

In [ ]:
example_id = 100
example = X_test.iloc[[example_id]]
true_value = y_test.iloc[example_id]
predicted_value = ridge_model.predict(example)[0]

print("Prawdziwa warto??:", dollars(true_value))
print("Predykcja Ridge:   ", dollars(predicted_value))
print("B??d:              ", dollars(abs(true_value - predicted_value)))

## Podsumowanie

Po tym notebooku umiesz:

- rozpozna? problem regresji,
- por?wna? model z baseline'em,
- u?y? `Pipeline` ze skalowaniem,
- wytrenowa? `LinearRegression` i `Ridge`,
- interpretowa? MAE, RMSE i R?,
- przeliczy? b??d modelu na praktyczn? jednostk?.